# Training the Models (BNNs, DNNs, BiLSTMs, and SNNs)

This notebook trains the neural network models on the MNIST dataset (and can be adapted for other datasets).

**Outputs**
- Trained model checkpoints (`.pth` files) containing the `model_state_dict`, and other training information, such as test accuracies and loss.

In [ ]:
# Copyright (c) 2026 Madelyn Cruz and Daniel Forger
# University of Michigan
# All rights reserved.

In [1]:
import torch
import matplotlib.pyplot as plt

from bnbp5.bnn_intralayer import *
from bnbp5.trainnospikemne_intralayer_timeind_sliding_128 import Trainer
from bnbp5.mnist_spiketrain_sliding import *
import numpy as np
from os.path import exists

from zanj import ZANJ

from torch import nn

torch.set_default_dtype(torch.float32)

In [ ]:
# Parameters for Hodgkin-Huxley neuron model
_HH_PARAMS: dict[str, float] = {
    "gna": 40.0,
    "gk": 35.0,
    "gl": 0.4, #0.3, 0.4

    "Ena": 55.0,
    "Ek": -77.0,
    "El": -65.0,

    "gm": 0.075,
    "ghca": 12.0, #changed 0.12,
    "vthresh": -56.2, 
    "tau_max": 0.608,
    "Eca":  55.0,#120.0,
    
    "gs": 0.04,
    "Vs": 0.0,
    "Iapp": 0.5,
    "lat_inhibition": False,
    "beta_n_modified": False,

    "Vt": -3.0,
    "Kp": 8.0,
    "a_d": 1.0,
    "a_r": 0.1    
}   

# Adjust for DNN/BiLSTM/SNN
# Running for MNIST Digits right now. For Anesthesia dataset or other datasets, adjust CFG1.model_dims e.g. model_dims = [28*28,100,2], dt = 1000/220/45
CFG1: BNNConfig = BNNConfig(
            neuron_model = model_HH_RS, lr = 0.0005, test_batch_sz = 25, train_batch_sz = 10, neuron_params = _HH_PARAMS, model_dims =[28*28,100,10])#, use_DNN = True)#, model_dims = [1, 1, 10]) 
    
    
#CFG1: BNNConfig = BNNConfig(lr = 0.0001, neuron_model = model_HH_IBN, test_batch_sz = 25, train_batch_sz = 20, neuron_params = _HH_PARAMS, model_dims = [128,100,2], dt = 1000/220/45)
    

CFG2: DatasetConfig = DatasetConfig(
            sim_t = 2000,n_samples_train=300, n_samples_val = 50, n_samples_test=9000)
    

torch.manual_seed(15)
    

trainer = Trainer(CFG1, CFG2, False, subjects=["UM_7"], num_classes=2, dataset = 'mnist')#, download=True)    

'''
trainer = Trainer(
    CFG1,
    CFG2,
    dataset="anesthesia",
    eeg_root='/nfs/turbo/lsa-forger/mccruz/Anesthesia',
    subjects=["UM_7"],
    num_classes=2,
    epoch_seconds=4.0,
    #train_stride_seconds=1.0,  # 75% overlap within training
    validation_fraction=0.3,
    test_fraction=0.20,
    gap_seconds=1.0,
    bandpass=(0.5, 60.0),
    seed=15,
)
'''

In [ ]:
# For pretrained models, change the .pth file name and uncomment this section.
'''
state = torch.load(  'BNN_Anesthesia_Sub007.pth',map_location=torch.device('cuda'))

model_state_dict = state['model_state_dict']
optimizer_state_dict = state['optimizer_state_dict']

trainer.model.load_state_dict(model_state_dict)
#trainer.optimizer.state_dict(optimizer_state_dict)

#for param_group in trainer.optimizer.param_groups:
#    param_group['lr'] = param_group['lr']*0.1
'''

In [ ]:
# Training the model for 20 epochs
training_accuracies_epochs= []#state['training_accuracies_epochs']
accuracies_epochs= []#state['accuracies_epochs']
loss_epochs = []# state['loss']
#epch = state['epoch']

# With accuracies and loss_record
trainer.optimizer.param_groups[0]['lr'] = 0.0005

for epch in range(20):
        print("epoch", epch)
        training_accuracies, loss_record = trainer.train(epch, 4)
        accuracies = trainer.test()

        training_accuracies_epochs.append(training_accuracies)
        accuracies_epochs.append(accuracies)
        loss_epochs.append(loss_record)

        print(accuracies_epochs)

        torch.save({
        'epoch': epch,
        'model_state_dict': trainer.model.state_dict(),
        'optimizer_state_dict': trainer.optimizer.state_dict(),
        'training_accuracies_epochs': training_accuracies_epochs,
        'accuracies_epochs': accuracies_epochs,
        'loss': loss_epochs,
        'CFG1':CFG1,
        'CFG2':CFG2,
        }, 'BNN_MNIST_RS.pth')#'BNN_Anesthesia_Sub007_2states.pth')


print("DONE")